# Correlation matrices + rates/asset scatter

1. **Level corr** — `CORR_SERIES`
2. **Corr percentile** — 100 = más alto en `CORR_PCT_LOOKBACK`
3. **Scatter** — x = ρ(USGG5YR, CO1), y = ρ(USGG5YR, ESA); **últimos 4 días en azul**

Knobs en `curve_config.py`: `CORR_SCATTER_RATES`, `CORR_SCATTER_X`, `CORR_SCATTER_Y`, `CORR_SCATTER_HIGHLIGHT_DAYS`.
Nuevos tickers (`USGG5YR`, `ESA`): `CROSS_ASSET` + `--pull-history`.


In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "correlations.py").exists() else _DZ
if not (REPO / "correlations.py").exists():
    raise FileNotFoundError("No encuentro correlations.py en %s — haz git pull" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import curve_config
import correlations

importlib.reload(curve_config)
importlib.reload(correlations)

print("CROSS_ASSET:", curve_config.CROSS_ASSET)
print("CORR_SERIES:", curve_config.CORR_SERIES)
print("window=%s  pct_lookback=%s" % (
    curve_config.CORR_WINDOW, curve_config.CORR_PCT_LOOKBACK,
))

In [ ]:
level, pmat, full, scatter_path, scatter_df = correlations.run_correlations()
print("panel cols sample:", list(full.columns)[:20], "...")
print("asof:", full.index[-1], "n=", len(full))
print("scatter:", scatter_path)


In [ ]:
# ── 1) Matriz de correlaciones (nivel) ───────────────────────────────────────
display(level.round(3))

In [ ]:
# ── 2) Percentil del rolling correlation (100 = máximo) + heatmap ────────────
display(pmat.round(1))

heat = Path(curve_config.GRAPH_DIR_CORR) / "corr_percentile_heatmap.png"
if heat.exists():
    display(Image(filename=str(heat)))
else:
    print("no heatmap — check matplotlib")


In [ ]:
# Level heatmap (opcional)
heat_l = Path(curve_config.GRAPH_DIR_CORR) / "corr_level_heatmap.png"
if heat_l.exists():
    display(Image(filename=str(heat_l)))

In [ ]:
# ── 3) Scatter ρ(rates, CO1) vs ρ(rates, ESA) — last 4 days in blue ──────────
heat_s = Path(curve_config.GRAPH_DIR_CORR) / "corr_scatter_rates_assets.png"
if heat_s.exists():
    display(Image(filename=str(heat_s)))
elif scatter_path and Path(scatter_path).exists():
    display(Image(filename=str(scatter_path)))
else:
    print("no scatter — need USGG5YR/CO1/ESA in panel (CROSS_ASSET + pull-history)")
if scatter_df is not None:
    display(scatter_df.tail(10))
